In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:

# -----------------------------
# Buckets + Colors (3 buckets, 3 colors)
# Unique global minimum: (R,G,B) for (B1,B2,B3)
# -----------------------------

# Encode colors as integers: R=0, G=1, B=2
idx2c = np.array(["R","G","B"])
B = 3
q = 3

# Unary costs: bucket i prefers its "target" color (cost 0), others cost 2
# This makes RGB uniquely optimal.
costs = np.array([
    [0, 2, 2],  # bucket 1 prefers R
    [2, 0, 2],  # bucket 2 prefers G
    [2, 2, 0],  # bucket 3 prefers B
], dtype=np.float64)

# Repeat penalty: discourages same color in multiple buckets
lambda_repeat = 3.0
pairs = np.array([(0,1),(0,2),(1,2)], dtype=np.int64)

In [3]:
def energy(cfg: np.ndarray) -> float:
    """
    cfg shape (B,), entries in {0,1,2}.
    E = E_pref + (lambda * E_rep)
    """
    E = costs[np.arange(B), cfg].sum()  # unary term
    E += lambda_repeat * np.sum(cfg[pairs[:,0]] == cfg[pairs[:,1]])  # repeat penalty
    return float(E)

In [4]:
def propose_neighbor(cfg: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """One-bucket move: change one bucket’s color to a different one."""
    i = rng.integers(0, B)
    old = cfg[i]
    delta = rng.integers(1, q)
    new = (old + delta) % q
    cand = cfg.copy()
    cand[i] = new
    return cand

In [5]:
def T_log(t: int, T0: float=8.0, c: float=2.0, Tmin: float=1e-4) -> float:
    """Log cooling: T(t)=T0/log(c+t), floored at Tmin."""
    return max(Tmin, T0 / np.log(c + t))

In [6]:
def simulated_annealing_log(steps=10000, T0=8.0, Tmin=1e-4, c=2.0, seed=7):
    rng = np.random.default_rng(seed)

    cfg = rng.integers(0, q, size=(B,), dtype=np.int64)
    E = energy(cfg)

    best_cfg = cfg.copy()
    best_E = E

    Ts = np.empty(steps, dtype=np.float64)
    Es = np.empty(steps, dtype=np.float64)
    bestEs = np.empty(steps, dtype=np.float64)

    for t in range(1, steps+1):
        T = T_log(t, T0=T0, c=c, Tmin=Tmin)

        cand = propose_neighbor(cfg, rng)
        Ecand = energy(cand)
        dE = Ecand - E

        # Metropolis accept rule
        if dE <= 0 or rng.random() < np.exp(-dE / T):
            cfg, E = cand, Ecand

        if E < best_E:
            best_cfg, best_E = cfg.copy(), E

        Ts[t-1] = T
        Es[t-1] = E
        bestEs[t-1] = best_E

    return best_cfg, best_E, Ts, Es, bestEs

In [7]:
# ---- run ----
best_cfg, best_E, Ts, Es, bestEs = simulated_annealing_log()
print("Best config:", "".join(idx2c[best_cfg]), " Best energy:", best_E)

Best config: RGB  Best energy: 0.0


In [11]:
"""
# ---- plot: energy trace ----
steps = np.arange(1, len(Es)+1)

plt.figure()
plt.plot(steps, Es, label="E(current)")
plt.plot(steps, bestEs, label="E(best-so-far)")
plt.xlabel("SA step")
plt.ylabel("Energy")
plt.legend()
plt.show()
"""

'\n# ---- plot: energy trace ----\nsteps = np.arange(1, len(Es)+1)\n\nplt.figure()\nplt.plot(steps, Es, label="E(current)")\nplt.plot(steps, bestEs, label="E(best-so-far)")\nplt.xlabel("SA step")\nplt.ylabel("Energy")\nplt.legend()\nplt.show()\n'

In [10]:
"""
# ---- plot: temperature schedule ----
plt.figure()
plt.plot(steps, Ts)
plt.xlabel("SA step")
plt.ylabel("Temperature T(t)")
plt.yscale("log")
plt.show()
"""

'\n# ---- plot: temperature schedule ----\nplt.figure()\nplt.plot(steps, Ts)\nplt.xlabel("SA step")\nplt.ylabel("Temperature T(t)")\nplt.yscale("log")\nplt.show()\n'

# Simulated Annealing for Buckets + Colors (Cliffnotes)

Goal

Find the lowest-energy configuration of 3 buckets, each choosing one of 3 colors ${R,G,B}$.

A configuration is $\sigma$ = ($\sigma_1,\sigma_2, \sigma_3$). We want the unique optimum (R,G,B).

1) Encode colors as integers
	•	Store colors as integers: 0=R, 1=G, 2=B.
	•	B = number of buckets (variables).
	•	q = number of states per bucket.

2) Define bucket preferences (unary costs)
	•	costs[i, c] = cost if bucket i chooses color c.
	•	Bucket 1 prefers R (cost 0), bucket 2 prefers G, bucket 3 prefers B.
	•	This makes RGB naturally the best.

3) Add “no repeats” penalty (pairwise constraint)

   • Penalize configurations where two buckets pick the same color.
   • Pairs lists all bucket pairs (i,j) to check equality.
   • cfg[pair_vec->col0] == cfg[pair_vec->col1] returns boolean array. for any repetitions, sum[bool_array] > 0. (pair_vec->col0 = pairs[:,0] and similarly col1 = pairs[:,1])

5) Energy function $E(\sigma)$

What it does:
	•	cfg is a length-3 array, like [0,1,2] meaning RGB.
	•	First line: adds unary costs $\sum_i c_i(\sigma_i)$.
	•	Second line: adds $\lambda$ for every repeated-color pair.
	•	Returns the scalar energy.

Lower energy = better.

5) Propose a neighbor configuration (one move)
Interpretation:
	•	Pick one bucket index i uniformly.
	•	Change only that bucket’s color.
	•	delta $\in {1,2}$ ensures you never pick the same color again.
	•	(old + delta) % 3 guarantees new != old.

This is the classic “single-spin flip” move in SA.

6) Logarithmic cooling schedule $T(t)$
   •	Temperature starts high, decreases slowly.
   •	Log cooling means $T(t)\propto 1/\log(t)$.
   •	Tmin is a floor so you don’t hit division/overflow weirdness.

High T: explore freely (accept worse moves).
Low T: become greedy (rarely accept worse moves).

7) Simulated annealing main loop
What’s happening:
	•	Initialize random configuration cfg.
	•	Track its energy E.
	•	Keep best_cfg and best_E because SA can temporarily move uphill.

Each step:
	1.	Compute temperature T(t).
	2.	Propose a neighbor cand.
	3.	Compute energy change \Delta E.
	4.	Metropolis accept rule:
	•	If $\Delta E \le 0$: accept
	•	Else accept with probability $e^{-\Delta E / T}$

Why this matters:
	•	At high T, you accept uphill moves often → escape local minima.
	•	At low T, uphill moves are rare → settle into low-energy basin.

Return:
	•	Best configuration ever seen, best energy,
	•	traces (Ts, Es, bestEs) for diagnostics.

8) Final readout
	•	Convert integers back to letters.
	•	For this toy, you should get RGB with energy 0.0.